## Bài 1


In [ ]:
"""Bài 1: tự viết KNN không dùng thư viện KNN."""

from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier


ROOT = Path.cwd()


def knn_du_doan(X_hoc, y_hoc, x_moi, k):
    """Dự đoán một mẫu mới bằng khoảng cách Euclid và bỏ phiếu đa số."""
    khoang_cach = np.sqrt(((X_hoc - x_moi) ** 2).sum(axis=1))
    chi_so_gan = np.argsort(khoang_cach)[:k]
    phieu = np.bincount(y_hoc[chi_so_gan])
    return int(np.argmax(phieu))


df = pd.read_csv(ROOT / "data" / "hai_vang_trang.csv")
X = df[["x1", "x2"]].to_numpy()
y = df["nhan"].to_numpy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

manual_prediction = np.array([
    knn_du_doan(X_train, y_train, sample, k=13) for sample in X_test
])
sklearn_model = KNeighborsClassifier(n_neighbors=13).fit(X_train, y_train)
sklearn_prediction = sklearn_model.predict(X_test)

print(f"Accuracy KNN tự viết : {accuracy_score(y_test, manual_prediction):.4f}")
print(f"Accuracy sklearn     : {accuracy_score(y_test, sklearn_prediction):.4f}")
print(f"Số dự đoán giống nhau: {(manual_prediction == sklearn_prediction).sum()}/{len(y_test)}")
print("\nGiải thích:")
print("- Với từng mẫu kiểm tra, hàm tính khoảng cách Euclid đến mọi mẫu học.")
print("- np.argsort lấy 13 khoảng cách nhỏ nhất; np.bincount đếm phiếu từng lớp.")
print("- Lớp có nhiều phiếu nhất là dự đoán. Quy trình này giống KNeighborsClassifier,")
print("  nên hai mô hình cho cùng 75/75 dự đoán và cùng accuracy.")


Accuracy KNN tự viết : 0.9200
Accuracy sklearn     : 0.9200
Số dự đoán giống nhau: 75/75

Giải thích:
- Với từng mẫu kiểm tra, hàm tính khoảng cách Euclid đến mọi mẫu học.
- np.argsort lấy 13 khoảng cách nhỏ nhất; np.bincount đếm phiếu từng lớp.
- Lớp có nhiều phiếu nhất là dự đoán. Quy trình này giống KNeighborsClassifier,
  nên hai mô hình cho cùng 75/75 dự đoán và cùng accuracy.


## Bài 2


In [ ]:
"""Bài 2: so sánh khoảng cách Manhattan và Euclid."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)


def chon_va_cham(metric):
    scores = {
        k: cross_val_score(
            KNeighborsClassifier(n_neighbors=k, metric=metric),
            X_train,
            y_train,
            cv=5,
        ).mean()
        for k in range(1, 52, 2)
    }
    best_k = max(scores, key=scores.get)
    model = KNeighborsClassifier(n_neighbors=best_k, metric=metric).fit(X_train, y_train)
    test_accuracy = accuracy_score(y_test, model.predict(X_test))
    return best_k, scores[best_k], test_accuracy


euclid = chon_va_cham("euclidean")
manhattan = chon_va_cham("manhattan")

print(f"{'Khoảng cách':<12} {'k tốt nhất':>12} {'CV':>8} {'test':>8}")
print(f"{'Euclid':<12} {euclid[0]:12d} {euclid[1]:8.4f} {euclid[2]:8.4f}")
print(f"{'Manhattan':<12} {manhattan[0]:12d} {manhattan[1]:8.4f} {manhattan[2]:8.4f}")
chenh_lech = manhattan[2] - euclid[2]
print(
    f"\nNhận xét: Manhattan chọn k = {manhattan[0]} thay vì {euclid[0]} và accuracy kiểm tra "
    f"thay đổi {chenh_lech:+.4f}. Mức chênh lệch nhỏ, nên đổi khoảng cách không làm kết quả khác nhiều."
)


Khoảng cách    k tốt nhất       CV     test
Euclid                 13   0.8667   0.9200
Manhattan              27   0.8667   0.9067

Nhận xét: Manhattan chọn k = 27 thay vì 13 và accuracy kiểm tra thay đổi -0.0133. Mức chênh lệch nhỏ, nên đổi khoảng cách không làm kết quả khác nhiều.


## Bài 3


In [ ]:
"""Bài 3: đổi đơn vị chi tiêu và kiểm tra tác động lên KNN."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "sinh_vien_mo_rong.csv")
y = df["qua_mon"]
columns = ["gio_on", "diem_giua_ky", "chi_tieu"]

X_nghin = df[columns].copy()
X_trieu = X_nghin.copy()
X_trieu["chi_tieu"] = X_trieu["chi_tieu"] / 1000

train_index, test_index = train_test_split(
    df.index, test_size=0.25, random_state=17, stratify=y
)
y_train, y_test = y.loc[train_index], y.loc[test_index]


def accuracy(X, standardized):
    model = (
        make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=7))
        if standardized
        else KNeighborsClassifier(n_neighbors=7)
    )
    model.fit(X.loc[train_index], y_train)
    return accuracy_score(y_test, model.predict(X.loc[test_index]))


print("Không chuẩn hóa:")
print(f"  Chi tiêu tính bằng nghìn đồng: {accuracy(X_nghin, False):.4f}")
print(f"  Chi tiêu tính bằng triệu đồng: {accuracy(X_trieu, False):.4f}")
print("Có chuẩn hóa:")
print(f"  Chi tiêu tính bằng nghìn đồng: {accuracy(X_nghin, True):.4f}")
print(f"  Chi tiêu tính bằng triệu đồng: {accuracy(X_trieu, True):.4f}")

print("\nGiải thích: khi chưa chuẩn hóa, đổi đơn vị làm thay đổi độ lớn của cột chi tiêu")
print("và vì thế làm thay đổi khoảng cách. Sau chuẩn hóa, hai cách ghi đơn vị tương đương.")


Không chuẩn hóa:
  Chi tiêu tính bằng nghìn đồng: 0.5333
  Chi tiêu tính bằng triệu đồng: 0.8667
Có chuẩn hóa:
  Chi tiêu tính bằng nghìn đồng: 0.8667
  Chi tiêu tính bằng triệu đồng: 0.8667

Giải thích: khi chưa chuẩn hóa, đổi đơn vị làm thay đổi độ lớn của cột chi tiêu
và vì thế làm thay đổi khoảng cách. Sau chuẩn hóa, hai cách ghi đơn vị tương đương.


## Bài 4


In [ ]:
"""Bài 4: so sánh bỏ phiếu đều và trọng số theo khoảng cách."""

from pathlib import Path

import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "hai_vang_trang.csv")
X = df[["x1", "x2"]]
y = df["nhan"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=17, stratify=y
)

print(f"{'k':>4} {'uniform':>10} {'distance':>10}")
for k in [1, 13, 101]:
    results = []
    for weights in ["uniform", "distance"]:
        model = KNeighborsClassifier(n_neighbors=k, weights=weights).fit(X_train, y_train)
        results.append(accuracy_score(y_test, model.predict(X_test)))
    print(f"{k:4d} {results[0]:10.4f} {results[1]:10.4f}")

print("\nVới k lớn, trọng số khoảng cách giúp các điểm gần có tiếng nói mạnh hơn,")
print("nên giảm tác động làm trơn quá mức của nhiều láng giềng ở xa.")


   k    uniform   distance
   1     0.8000     0.8000
  13     0.9200     0.9333
 101     0.8533     0.9067

Với k lớn, trọng số khoảng cách giúp các điểm gần có tiếng nói mạnh hơn,
nên giảm tác động làm trơn quá mức của nhiều láng giềng ở xa.


## Bài 5


In [ ]:
"""Bài 5: chọn k cho KNN hồi quy bằng kiểm định chéo."""

from pathlib import Path

import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor


ROOT = Path.cwd()
df = pd.read_csv(ROOT / "data" / "gia_nha.csv")
X = df[["dien_tich"]]
y = df["gia"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

cv_mae = {}
for k in range(1, 21):
    scores = cross_val_score(
        KNeighborsRegressor(n_neighbors=k),
        X_train,
        y_train,
        cv=5,
        scoring="neg_mean_absolute_error",
    )
    cv_mae[k] = -scores.mean()

best_k = min(cv_mae, key=cv_mae.get)
knn = KNeighborsRegressor(n_neighbors=best_k).fit(X_train, y_train)
linear = LinearRegression().fit(X_train, y_train)
mae_knn = mean_absolute_error(y_test, knn.predict(X_test))
mae_linear = mean_absolute_error(y_test, linear.predict(X_test))

print("MAE kiểm định chéo theo k:")
for k in range(1, 21):
    note = " <- tốt nhất" if k == best_k else ""
    print(f"  k = {k:2d}: {cv_mae[k]:.4f}{note}")

print(f"\nk tốt nhất = {best_k}")
print(f"MAE KNN trên tập kiểm tra       : {mae_knn:.4f}")
print(f"MAE đường thẳng trên tập kiểm tra: {mae_linear:.4f}")
if mae_knn < mae_linear:
    print("KNN vượt đường thẳng trên tập kiểm tra.")
else:
    print("KNN chưa vượt đường thẳng vì quan hệ giá và diện tích gần tuyến tính.")


MAE kiểm định chéo theo k:
  k =  1: 0.4164
  k =  2: 0.3498 <- tốt nhất
  k =  3: 0.3714
  k =  4: 0.3618
  k =  5: 0.3816
  k =  6: 0.4002
  k =  7: 0.3990
  k =  8: 0.4098
  k =  9: 0.4171
  k = 10: 0.4365
  k = 11: 0.4500
  k = 12: 0.4695
  k = 13: 0.4992
  k = 14: 0.5153
  k = 15: 0.5290
  k = 16: 0.5595
  k = 17: 0.5748
  k = 18: 0.5891
  k = 19: 0.6130
  k = 20: 0.6254

k tốt nhất = 2
MAE KNN trên tập kiểm tra       : 0.4867
MAE đường thẳng trên tập kiểm tra: 0.2578
KNN chưa vượt đường thẳng vì quan hệ giá và diện tích gần tuyến tính.


## Bài 6


In [ ]:
"""Bài 6: đo thời gian fit và predict của KNN khi dữ liệu tăng."""

from time import perf_counter

import numpy as np
from sklearn.neighbors import KNeighborsClassifier


rng = np.random.default_rng(0)
X_new = rng.random((1000, 2))

print(f"{'Số mẫu học':>12} {'fit (giây)':>12} {'predict 1000 TB (giây)':>25}")
for n_samples in [1000, 10000, 100000]:
    X_train = rng.random((n_samples, 2))
    y_train = rng.integers(0, 2, size=n_samples)
    # Dùng tìm kiếm vét cạn để quan sát rõ chi phí so sánh với toàn bộ tập học.
    model = KNeighborsClassifier(n_neighbors=5, algorithm="brute", n_jobs=-1)

    start = perf_counter()
    model.fit(X_train, y_train)
    fit_time = perf_counter() - start

    # Khởi động thư viện tính toán trước khi đo để lần đầu không tính thời gian nạp hệ thống.
    model.predict(X_new[:10])
    start = perf_counter()
    for _ in range(5):
        model.predict(X_new)
    predict_time = (perf_counter() - start) / 5

    print(f"{n_samples:12d} {fit_time:12.6f} {predict_time:25.6f}")

print("\nKNN được gọi là học lười vì fit chủ yếu chỉ lưu dữ liệu nên rất nhanh.")
print("Khi predict, mô hình phải tìm hàng xóm trong tập học; dữ liệu càng lớn thì dự đoán càng lâu.")
print("Hai cỡ nhỏ có thể gần nhau do chi phí khởi động; cỡ 100000 cho thấy chi phí dự đoán tăng rõ.")
print("Thời gian cụ thể có thể khác nhau giữa các máy, cần so sánh xu hướng tương đối.")


  Số mẫu học   fit (giây)    predict 1000 TB (giây)
        1000     0.000476                  0.036480
       10000     0.001325                  0.042274
      100000     0.008135                  0.283913

KNN được gọi là học lười vì fit chủ yếu chỉ lưu dữ liệu nên rất nhanh.
Khi predict, mô hình phải tìm hàng xóm trong tập học; dữ liệu càng lớn thì dự đoán càng lâu.
Hai cỡ nhỏ có thể gần nhau do chi phí khởi động; cỡ 100000 cho thấy chi phí dự đoán tăng rõ.
Thời gian cụ thể có thể khác nhau giữa các máy, cần so sánh xu hướng tương đối.
